# NovaBank Interpretability And Model Risk

This notebook turns the v0.4 digital-banking model outputs into inspectable explanations and a model-risk review for **NovaBank Digital**. It is the second of two v0.7 interpretability track notebooks and consumes the explanation utilities (slice 1), the threshold / false-positive utilities (slice 2), and the model-documentation template (slice 3).

Learning goal: explain *why* a specific digital alert was generated, separate rule, model, graph, and case signal types, choose a threshold that reflects alert capacity and cost, review where false positives concentrate, and keep the limitation-aware framing visible — **a model should not be judged by headline accuracy**.

> Educational exercise on synthetic data. No real Client, account, or transaction data; no certification or legal-advice claim. The **User** is the digital login identity; the **Client** is the legal customer.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from banking_fraud_lab import (
    build_digital_banking_features,
    build_learner_facing_views,
    build_model_documentation,
    build_partial_dependence_grid,
    concentrate_false_positives,
    evaluate_alert_scores,
    extract_feature_importance,
    generate_digital_fraud_scenarios_world,
    recommend_lowest_cost_threshold,
)
from banking_fraud_lab.features import DIGITAL_BANKING_FEATURE_FAMILIES
from banking_fraud_lab.interpretability import PATTERN_TO_EXPLANATION_FAMILY
from banking_fraud_lab.schema import PROTECTED_SCENARIO_ANSWER_KEYS

pd.set_option("display.max_columns", 40)

## Generate Learner-Facing Data

The supervised label comes from generated case outcomes for the digital `new_beneficiary_payment`, `session_payment_velocity`, and `digital_scam_to_mule` Detection patterns.

In [ ]:
tables = generate_digital_fraud_scenarios_world(
    seed=42,
    scale="small",
    scenario_prevalence=0.5,
    noisy_outcome_rate=0.3,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

learner_summary = pd.DataFrame(
    {
        "table": ["cases", "case_outcomes", "alerts", "transactions"],
        "rows": [
            len(learner_tables["cases"]),
            len(learner_tables["case_outcomes"]),
            len(learner_tables["alerts"]),
            len(learner_tables["transactions"]),
        ],
    }
)
learner_summary

## Build The Supervised Modeling Frame

Join cases, alerts, outcomes, and the v0.4 digital feature frame, then fit the same reproducible baseline used in the supervised baseline notebook so explanations are anchored to a real fitted model.

In [ ]:
feature_frame = build_digital_banking_features(learner_tables)

model_frame = (
    learner_tables["cases"][["case_id", "alert_id", "transaction_id"]]
    .merge(
        learner_tables["alerts"][["alert_id", "alert_type", "severity", "reason"]],
        on="alert_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        learner_tables["case_outcomes"][["case_id", "confirmed_fraud", "loss_amount_chf"]],
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(feature_frame, on="transaction_id", how="inner", validate="one_to_one")
)
assert model_frame["confirmed_fraud"].nunique() == 2

feature_output_columns = [
    output_column
    for spec in DIGITAL_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
]
numeric_feature_columns = [
    column
    for column in feature_output_columns
    if pd.api.types.is_numeric_dtype(model_frame[column])
]
assert numeric_feature_columns

preprocess = ColumnTransformer(
    [("numeric", StandardScaler(), numeric_feature_columns)],
    remainder="drop",
)
baseline_model = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "model",
            LogisticRegression(
                random_state=42,
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
            ),
        ),
    ]
)
target = model_frame["confirmed_fraud"].astype(bool)
baseline_model.fit(model_frame[numeric_feature_columns], target)

model_frame = model_frame.assign(
    score=baseline_model.predict_proba(
        model_frame[numeric_feature_columns]
    )[:, 1].round(6)
)
model_frame[["alert_id", "alert_type", "confirmed_fraud", "score"]].head()

## Why Was This Alert Generated? Per-Alert Explanation

Feature importance explains which inputs drove each digital alert's score. The `ExplanationFamilySpec` vocabulary ties every feature to its **Detection pattern** (e.g. `digital_scam_to_mule`, `new_beneficiary_payment`, `session_payment_velocity`) so the explanation is traceable, not a black box. Importance is an inspection aid, not proof the model is correct — a model should not be judged by headline accuracy.

In [ ]:
scam_spec = PATTERN_TO_EXPLANATION_FAMILY["digital_scam_to_mule"]
beneficiary_spec = PATTERN_TO_EXPLANATION_FAMILY["new_beneficiary_payment"]

# Feature importance is global to the fitted model; a family groups a subset of the
# model's columns, so we score over ALL numeric features and then filter to each
# family's columns to read that pattern's drivers.
global_importance = extract_feature_importance(
    baseline_model,
    numeric_feature_columns,
    feature_frame=model_frame[numeric_feature_columns],
)

importance_overview = pd.concat(
    [
        global_importance[
            global_importance["feature"].isin(scam_spec.feature_columns)
        ].assign(detection_pattern_id="digital_scam_to_mule"),
        global_importance[
            global_importance["feature"].isin(beneficiary_spec.feature_columns)
        ].assign(detection_pattern_id="new_beneficiary_payment"),
    ]
)
importance_overview.sort_values("native_importance", ascending=False)

### Partial-Dependence Grid For The Top Driver

A partial-dependence / ICE grid shows how the positive-class score moves as one digital feature is swept across its observed range — a compact view of the model's marginal behaviour, tied to the same Detection pattern id.

In [ ]:
top_feature = str(
    importance_overview.sort_values("native_importance", ascending=False)
    .iloc[0]["feature"]
)
pd_grid = build_partial_dependence_grid(
    baseline_model,
    model_frame[numeric_feature_columns],
    top_feature,
    grid_points=7,
    detection_pattern_id="digital_scam_to_mule",
)
pd_grid

## Compare Rule, Model, Graph, And Case Evidence

Digital-banking alerts draw on four signal types. Keeping them side by side lets a reviewer separate signal types: a model score, a deterministic rule trigger, the v0.6 **graph** signal (`mule_ring` / `digital_scam_to_mule` network), and the v0.5 **case** narrative. Each adds something the others do not.

| Signal type | v0.x source | What it adds to a digital alert |
|---|---|---|
| Rule | v0.1 alert generator | Deterministic trigger thresholds (velocity, beneficiary age). |
| Model | v0.4 supervised baseline | Calibrated score from session/beneficiary/pass-through features. |
| Graph | v0.6 `mule_ring` / `digital_scam_to_mule` | Shared-device rings, pass-through clusters. |
| Case | v0.5 NovaBank narrative | Investigator reasoning and scam outcome. |

In [ ]:
signal_evidence = pd.DataFrame(
    [
        {
            "alert_id": row["alert_id"],
            "alert_type": row["alert_type"],
            "model_score": row["score"],
            "rule_triggered": row["alert_type"]
            in {
                "new_beneficiary_payment",
                "session_payment_velocity",
                "digital_scam_to_mule_flow",
            },
            "graph_pattern": "mule_ring"
            if row["alert_type"] == "digital_scam_to_mule_flow"
            else None,
        }
        for _, row in model_frame.head(8).iterrows()
    ]
)
signal_evidence

## Threshold Selection With Capacity And Costs

Thresholds are an operational decision: a low threshold raises recall but overloads investigators and inflates false-positive cost. The recommender sweeps alert capacity and the investigation / false-positive / missed-fraud costs so the chosen threshold reflects those tradeoffs rather than a default.

In [ ]:
alert_scores = pd.DataFrame(
    {"alert_id": model_frame["alert_id"], "score": model_frame["score"]}
)
threshold_recommendation = recommend_lowest_cost_threshold(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    candidate_thresholds=(0.25, 0.5, 0.75),
    alert_capacities=(5, 10, 25),
    investigation_cost_chf=75.0,
    false_positive_cost_chf=25.0,
    missed_fraud_cost_chf=1_000.0,
)

recommendation_overview = pd.DataFrame(
    [
        {
            "alert_capacity": capacity,
            "lowest_cost_threshold": entry["lowest_cost_threshold"],
            "total_cost_chf": entry["lowest_cost_summary"]["total_cost_chf"],
            "recall": entry["lowest_cost_summary"]["recall"],
        }
        for capacity, entry in threshold_recommendation["per_capacity"].items()
    ]
)
recommendation_overview

## False-Positive Concentration Review

Where do false positives fall? Grouping by `alert_type` shows whether review burden concentrates on one digital pattern — a review prompt, not a fairness verdict.

In [ ]:
fp_concentration = concentrate_false_positives(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    threshold=threshold_recommendation["recommended_threshold"],
    segment_columns=["alert_type"],
    alerts=learner_tables["alerts"],
)
fp_concentration

## Model Documentation

The frozen template fills every required section (purpose, data lineage, assumptions, limitations, monitoring needs) deterministically from the fitted model, producing a stakeholder-readable artifact for the governance memo.

In [ ]:
documentation = build_model_documentation(
    baseline_model,
    institution="NovaBank Digital",
    detection_pattern_id="digital_scam_to_mule",
    feature_columns=numeric_feature_columns,
    model_frame=model_frame,
    seed=42,
    scale="small",
    cost_parameters={
        "investigation_cost_chf": 75.0,
        "false_positive_cost_chf": 25.0,
        "missed_fraud_cost_chf": 1_000.0,
    },
)

documentation_sections = pd.DataFrame(
    [
        {
            "section": section["display_name"],
            "text": section["text"],
        }
        for section in documentation["sections"].values()
    ]
)
documentation_sections

## Keep Evaluation Limits Visible

Headline accuracy is out of scope: fraud labels are sparse, alert outcomes are operational decisions, and protected answer keys stay separate from learner-facing data. Explanations are inspection aids, not proof of correctness. **A model should not be judged by headline accuracy.**

In [ ]:
report = evaluate_alert_scores(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    thresholds=(threshold_recommendation["recommended_threshold"],),
    alert_capacity=10,
    investigation_cost_chf=75.0,
    false_positive_cost_chf=25.0,
    missed_fraud_cost_chf=1_000.0,
)
assert "accuracy is out of scope" in report["limitation_summary"]
pd.DataFrame(
    [
        {
            "metric": "limitation_summary",
            "value": report["limitation_summary"],
        }
    ]
)

## Optional SHAP Explanation

SHAP is an optional explainability tool, kept behind the `shap` extra so it does not add dependency cost to the core curriculum or CI. When the extra is installed, the cell below shows a SHAP-based feature ranking; when it is absent, the cell prints a skip message and the notebook continues. SHAP is a complementary view, not a requirement.

In [ ]:
from banking_fraud_lab import SHAP_AVAILABLE, explain_with_shap

if SHAP_AVAILABLE:
    shap_explanation = explain_with_shap(
        baseline_model,
        model_frame[numeric_feature_columns],
        numeric_feature_columns,
        detection_pattern_id="digital_scam_to_mule",
    )
    print(shap_explanation)
else:
    print(
        "SHAP optional extra not installed; skipping SHAP explanation. "
        "Install with `uv sync --extra shap` to view SHAP rankings."
    )